In [1]:
%%capture

%pip uninstall onnxruntime
%pip install --upgrade onnxruntime-gpu onnxscript
%pip install --upgrade transformers

In [32]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
from torch.export import Dim

In [33]:
%%capture

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "pameydorke/redred-gemma-4-E2B-it"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

In [36]:
# Quick check
print([a for a in dir(model.model) if 'vision' in a.lower()])
# Usually: 'vision_model'

['embed_vision', 'vision_tower']


In [38]:
model.model.vision_tower

Gemma4VisionModel(
  (patch_embedder): Gemma4VisionPatchEmbedder(
    (input_proj): Linear(in_features=768, out_features=768, bias=False)
  )
  (encoder): Gemma4VisionEncoder(
    (rotary_emb): Gemma4VisionRotaryEmbedding()
    (layers): ModuleList(
      (0-15): 16 x Gemma4VisionEncoderLayer(
        (self_attn): Gemma4VisionAttention(
          (q_proj): Gemma4ClippableLinear(
            (linear): Linear(in_features=768, out_features=768, bias=False)
          )
          (k_proj): Gemma4ClippableLinear(
            (linear): Linear(in_features=768, out_features=768, bias=False)
          )
          (v_proj): Gemma4ClippableLinear(
            (linear): Linear(in_features=768, out_features=768, bias=False)
          )
          (o_proj): Gemma4ClippableLinear(
            (linear): Linear(in_features=768, out_features=768, bias=False)
          )
          (q_norm): Gemma4RMSNorm()
          (k_norm): Gemma4RMSNorm()
          (v_norm): Gemma4RMSNorm()
        )
        (mlp): Gemm

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.export import Dim

# ------------------------------------------------------------------
# Vision wrapper
# ------------------------------------------------------------------
class VisionEncoderWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision_model = model.model.vision_tower  # Gemma4VisionModel

    def forward(self, pixel_values: torch.FloatTensor):
        # pixel_values: [B, 3, H, W] — Gemma 4 uses dynamic resolution
        out = self.vision_model(pixel_values, return_dict=True)
        # Returns [B, num_vision_tokens, vision_hidden_size]
        return out.last_hidden_state

vision_wrapper = VisionEncoderWrapper(model).cpu().eval()

# ------------------------------------------------------------------
# Dummy vision input (Gemma 4 accepts variable resolution)
# ------------------------------------------------------------------
B = 1
# Gemma 4 vision uses patch_size=16, so any multiple of 16 works
H, W = 336, 336  
dummy_pixels = torch.randn(B, 3, H, W, dtype=torch.float32)

# ------------------------------------------------------------------
# Export
# ------------------------------------------------------------------
batch_dim = Dim("batch")
h_dim = Dim("height")
w_dim = Dim("width")

torch.onnx.export(
    vision_wrapper,
    (dummy_pixels,),
    "vision_encoder.onnx",
    dynamo=True,
    opset_version=21,
    input_names=["pixel_values"],
    output_names=["vision_features"],
    dynamic_shapes=(
        {0: batch_dim, 2: h_dim, 3: w_dim},  # pixel_values
    ),
    external_data=True,
)

[torch.onnx] Obtain model graph for `VisionEncoderWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VisionEncoderWrapper([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `VisionEncoderWrapper([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `VisionEncoderWrapper([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'TypeError'>: Gemma4VisionModel.forward() missing 1 required positional argument: 'pixel_position_ids'

(Refer to the full stack trace above for more information.)

In [29]:
class ManualGemma4Decoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        # Grab the actual text decoder guts
        self.embed_tokens = model.model.language_model.embed_tokens
        self.layers       = model.model.language_model.layers
        self.norm         = model.model.language_model.layers
        self.lm_head      = model.lm_head

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: torch.LongTensor,
        position_ids: torch.LongTensor,
        past_key_values: tuple,
    ):
        # 1. Embeddings
        hidden_states = self.embed_tokens(input_ids)

        # 2. Build cache object from tuple
        cache = DynamicCache()
        cache.key_cache   = [kv[0] for kv in past_key_values]
        cache.value_cache = [kv[1] for kv in past_key_values]

        # 3. Hook through every decoder layer manually
        presents = []
        for i, layer in enumerate(self.layers):
            # Gemma4DecoderLayer forward signature
            layer_outputs = layer(
                hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=cache,          # passes the whole cache; layer i pulls its slice
                use_cache=True,
                output_attentions=False,
            )
            hidden_states = layer_outputs[0]
            # layer_outputs[1] is the updated cache (same object, but we grab the tensors)
            presents.append((
                cache.key_cache[i].clone(),
                cache.value_cache[i].clone(),
            ))

        # 4. Final norm + LM head
        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)

        return logits, tuple(presents)

In [30]:
model = model.cpu().eval()
wrapper = ManualGemma4Wrapper(model).cpu().eval()

In [22]:
config     = model.config
text_cfg   = config.text_config 

num_layers   = text_cfg.num_hidden_layers        # 35
num_kv_heads = text_cfg.num_key_value_heads      # 1
head_dim     = text_cfg.head_dim                 # 256
hidden_size  = text_cfg.hidden_size              # 1536
vocab_size   = text_cfg.vocab_size

In [23]:
B, S, P = 1, 1, 0

dummy_input_ids      = torch.randint(0, text_cfg.vocab_size, (B, S), dtype=torch.long)
dummy_attention_mask = torch.ones(B, P + S, dtype=torch.long)
dummy_position_ids   = torch.arange(P, P + S, dtype=torch.long).unsqueeze(0)

# Always pass tensors, never None, so Dynamo doesn’t hit control-flow branches.
# Shape per layer: [batch, num_kv_heads, past_len, head_dim]
dummy_past_kv = tuple(
    (
        torch.zeros(B, num_kv_heads, P, head_dim, dtype=torch.float32),
        torch.zeros(B, num_kv_heads, P, head_dim, dtype=torch.float32),
    )
    for _ in range(num_layers)
)

In [24]:
dummy_input_ids      = dummy_input_ids.cpu()
dummy_attention_mask = dummy_attention_mask.cpu()
dummy_position_ids   = dummy_position_ids.cpu()
dummy_past_kv        = tuple(
    (k.cpu(), v.cpu())
    for k, v in dummy_past_kv
)

In [25]:

batch_dim = Dim("batch")
seq_dim   = Dim("seq_len")
total_dim = Dim("total_len")
past_dim  = Dim("past_len")   # dynamic cache length

# Match the structure of (input_ids, attention_mask, position_ids, past_key_values)
dynamic_shapes = (
    {0: batch_dim, 1: seq_dim},           # input_ids
    {0: batch_dim, 1: total_dim},         # attention_mask
    {0: batch_dim, 1: seq_dim},           # position_ids
    tuple(                                # past_key_values: tuple of 35 layers
        (
            {0: batch_dim, 2: past_dim},  # key tensor
            {0: batch_dim, 2: past_dim},  # value tensor
        )
        for _ in range(num_layers)
    ),
)

In [31]:
torch.onnx.export(
    wrapper,
    (dummy_input_ids, dummy_attention_mask, dummy_position_ids, dummy_past_kv),
    "decoder_model_merged.onnx",
    dynamo=True,                # modern PyTorch exporter
    opset_version=21,
    input_names=["input_ids", "attention_mask", "position_ids", "past_key_values"],
    output_names=["logits", "present_key_values"],
    dynamic_shapes=dynamic_shapes,
    external_data=True,         # 4B model > 2 GB, so weights live alongside
)

[torch.onnx] Obtain model graph for `ManualGemma4Wrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ManualGemma4Wrapper([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `ManualGemma4Wrapper([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `ManualGemma4Wrapper([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'TypeError'>: cannot unpack non-iterable NoneType object

(Refer to the full stack trace above for more information.)